# 01 — REF concepts

This notebook introduces the core vocabulary of the **Rapid Evaluation Framework (REF)**.
By the end you will know what a *diagnostic*, *provider*, *execution*, *metric* and *dataset* are, and how they fit together.

**Prerequisites:** none. This is the place to start.

**What you need:** an internet connection as we read from the public REF API.

## What is the REF?

The REF performs evaluation of climate datasets, much like a CI/CD pipeline runs tests against code. 
As new climate model output is published, the REF evaluates it against reference data 
and produces figures and metrics — in near-real time.

The public deployment currently evaluates CMIP6 dataset, but will include CMIP7 Assessment Fast Track data
as they become available.

The REF has a dashboard (<https://dashboard.climate-ref.org>) which offers a curated view of the results from the REF. 
This dashboard uses an API (<https://api.climate-ref.org>) to provide access to the results in a structured way.

We can use this same API to make our own view of the data. As part of the API we also publish a [schema](https://api.climate-ref.org/docs) that describes the different endpoints that can be queried and the shape of the response

Throughout these notebooks we talk to the API via an "SDK" which is a small software package generated from the schema of the API.
This SDK allows us to make requests to the API as function calls without having to directly make HTTP requests.

A small script `scripts/generate_client.sh` shows how to built the SDK.

> If you are runnning this on Binder, this has been pre-built for you.

In [7]:
from ref_tutorials import get_client

client = get_client()
client

Client(raise_on_unexpected_status=False, _base_url='https://api.climate-ref.org', _cookies={}, _headers={}, _timeout=None, _verify_ssl=True, _follow_redirects=False, _httpx_args={}, _client=None, _async_client=None)

## Datasets

Datasets containing CMORised climate model output, and the reference data it is compared to are a key input to the REF.
The REF tracks which datasets are available and which diagnostics have already been run.

## Diagnostics

A **diagnostic** is a single, well-defined evaluation to understand a component of the earth system.
For example "the global mean surface temperature timeseries" or "the Atlantic overturning circulation strength".

Let's list the diagnostics the REF currently provides:

In [8]:
from climate_ref_client.api.diagnostics import diagnostics_list

diagnostics = diagnostics_list.sync(client=client).data
print(f"{len(diagnostics)} diagnostics available. Showing the first 10:\n")
for diagnostic in sorted(diagnostics, key=lambda d: d.name)[:10]:
    print(f"  {diagnostic.slug:35s} {diagnostic.name}")

47 diagnostics available. Showing the first 10:

  annual-cycle                        Annual Cycle Analysis
  sea-ice-area-basic                  Arctic and Antarctic Sea Ice Area Seasonal Cycle
  amoc-rapid                          Atlantic Meridional Overturning Circulation (RAPID)
  burntfractionall-gfed               Burnt Fraction (GFED)
  climate-drivers-for-fire            Climate Drivers for Fire
  climate-at-global-warming-levels    Climate at Global Warming Levels
  cloud-radiative-effects             Cloud Radiative Effects
  cloud-scatterplots-reference        Cloud Scatterplots for Reference dataset
  cloud-scatterplots-clwvi-pr         Cloud-Precipitation Scatterplots (clwvi vs pr)
  cloud-scatterplots-clivi-lwcre      Cloud-Radiation Scatterplots (clivi vs lwcre)


Every diagnostic has a human-readable `name`, a short machine-friendly `slug`, and a
`description`. Let's look at one in full:

In [9]:
example = diagnostics[0]
print("name:       ", example.name)
print("slug:       ", example.slug)
print("description:", example.description.strip())

name:        Climate at Global Warming Levels
slug:        climate-at-global-warming-levels
description: Calculate climate variables at global warming levels (e.g. 1.5C, 2C, 3C, 4C above pre-industrial).


## Providers

Diagnostics are grouped into **providers**. A **provider** is a package that knows how to compute a family of diagnostics.

For the Assessment Fast Track we use:

- ESMValTool
- PCMDI Metrics Package (PMP)
- ILAMB/IOMB

The REF orchestrates providers and does not compute diagnostics itself.
Each provider is a thin wrapper around the upstream diagnostics package.

Each diagnostic tells you which provider it belongs to:

In [10]:
providers = {d.provider.slug: d.provider.name for d in diagnostics}
for slug, name in sorted(providers.items()):
    count = sum(1 for d in diagnostics if d.provider.slug == slug)
    print(f"  {slug:20s} {name}  ({count} diagnostics)")

  esmvaltool           ESMValTool  (24 diagnostics)
  ilamb                ILAMB  (13 diagnostics)
  pmp                  PMP  (10 diagnostics)


## Execution groups and Executions

An **execution** is one run of a diagnostic against one specific group of datasets.
This execution has information about the datasets that were used to run it, the diagnostic and the outputs.

A single diagnostic is typically executed many times on different sets of data.
This generally depends what is being calculated, but this is often once per model variant.
Each of these subsets is an *execution group*.

Each execution group has a unique key that describes it.
If any datasets in an execution group change, then a new execution is performed.

We can list the execution groups available for the Global Warming Level diagnostic:

In [11]:
from climate_ref_client.api.diagnostics import diagnostics_list_execution_groups

execution_groups = diagnostics_list_execution_groups.sync(
    client=client, provider_slug=example.provider.slug, diagnostic_slug=example.slug
).data

first_execution_group = execution_groups[0]

print(f"Diagnostic: {example.name}")
print(f"Execution groups: {len(execution_groups)}")
print(f"Execution group keys: {[eg.key for eg in execution_groups]}")

Diagnostic: Climate at Global Warming Levels
Execution groups: 4
Execution group keys: ['cmip6_ssp126', 'cmip6_ssp245', 'cmip6_ssp370', 'cmip6_ssp585']


Within each execution group, there may be multiple executions.
For this execution we can access the outputs that were generated as well as a URL for downloading them.

In [12]:
for o in first_execution_group.latest_execution.outputs:
    print(f"  {o.filename:20s} {o.url}")

  executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_mean_1.5.png https://api.climate-ref.org/api/v1/results/267670
  executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_mean_2.0.png https://api.climate-ref.org/api/v1/results/267671
  executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_stdev_1.5.png https://api.climate-ref.org/api/v1/results/267672
  executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_stdev_2.0.png https://api.climate-ref.org/api/v1/results/267673
  executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_mean_1.5.png https://api.climate-ref.org/api/v1/results/267674
  executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_mean_2.0.png https://api.climate-ref.org/api/v1/results/267675
  executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_stdev_1.5.png https://api.cli

## Metrics

An execution produces output. That output comes in two main shapes:

- **Metric values** — numbers that summarise a property of a model. 
  A *scalar* is a single number (e.g. a root-mean-square error) or a *series* is a number per index point
  (e.g. a seasonal cycle).
- **Files** — NetCDF data and figures for deeper analysis or custom plotting.

The REF ingests all of these outputs into its database so they can be queried by the API.

Notebook 02 shows how to retrieve metric values, and notebook 03 turns them into a figure.

## Recap

| Term                | Meaning                                                                       |
| ------------------- | ----------------------------------------------------------------------------- |
| **Provider**        | A package that computes a family of diagnostics (ESMValTool, PMP, ILAMB, ...) |
| **Diagnostic**      | A single well-defined evaluation                                              |
| **Dataset**         | Climate model output or reference data a diagnostic runs against              |
| **Execution**       | One run of a diagnostic against one group of datasets                         |
| **Metric value**    | A scalar or series result summarising a model                                 |

**Next:** [02 — Querying the REF API](02-querying-the-api.ipynb).